# Whisper 모델 학습

## 1. 데이터 준비

In [1]:
import pandas as pd

In [3]:
# 전처리된 데이터(csv 파일) 불러오기
train_data = pd.read_csv(f"./dataset/dataset_train.csv") # 학습 데이터
val_data = pd.read_csv(f"./dataset/dataset_val.csv") # 검증 데이터

print(f"Train Data: {len(train_data)} samples")
print(f"Validation Data: {len(val_data)} samples")

Train Data: 628591 samples
Validation Data: 83695 samples


In [4]:
train_data.head()

,id,predText,labelText,gender,age,region,dialect
0,script1_p_0089-14002-02-01-KIS-F-08-D,희연재 말래시는 단계 체크에서 알려주겠나,현재 빨래 진행 단계 체크해서 알려 주겠나?,Female,60~69,부산/대구/울산/경상,경상
1,script1_p_0089-14003-02-01-KIS-F-08-D,유수 석부 궁금하네,뉴스 속보 궁금하네.,Female,60~69,부산/대구/울산/경상,경상
2,script1_p_0089-14004-02-01-KIS-F-08-D,이 약 보경법에 대해서 금세게 볼래.,이 약 복용법에 대해서 검색해 볼래?,Female,60~69,부산/대구/울산/경상,경상
3,script1_p_0089-14006-02-01-KIS-F-08-D,졸린이가 뒤로 있구자,졸리니까 티브이 끄자.,Female,60~69,부산/대구/울산/경상,경상
4,script1_p_0089-14007-02-01-KIS-F-08-D,6번 채널 로카카온,육 번 채널 녹화 가능한가?,Female,60~69,부산/대구/울산/경상,경상


In [5]:
val_data.head()

,id,predText,labelText,gender,age,region,dialect
0,n_0310-12001-02-01-BHJ-F-08-A,운동으로 체조하려는데 도와주면 좋겠네.,운동으로 체조 하려는데 도와 주면 좋겠네.,Female,60~69,서울/인천/경기,경기/서울
1,n_0310-12002-02-01-BHJ-F-08-A,알람 그만 올려줘,알람 그만 울려 줘.,Female,60~69,서울/인천/경기,경기/서울
2,n_0310-12003-02-01-BHJ-F-08-A,내가 따라 할 만한 건강체조 들어봐,내가 따라 할 만한 건강 체조 틀어 봐.,Female,60~69,서울/인천/경기,경기/서울
3,n_0310-12004-02-01-BHJ-F-08-A,나 땡빨롤에 부를게,나 땡벌 노래 부를게.,Female,60~69,서울/인천/경기,경기/서울
4,n_0310-12005-02-01-BHJ-F-08-A,동네 맛집 찾아서 나한테 알려줘,동네 맛집 찾아서 나한테 알려 줘.,Female,60~69,서울/인천/경기,경기/서울


In [ ]:
import pandas as pd
import torch
from datasets import Dataset
from transformers import WhisperProcessor, WhisperForConditionalGeneration, TrainingArguments, Trainer

# 1. 데이터 로드
data_path = "data.csv"  # CSV 파일 경로
df = pd.read_csv(data_path)

# 2. 데이터셋 변환
def preprocess_function(examples):
    return processor(
        examples["predText"],
        return_tensors="pt",
        padding="max_length",
        truncation=True
    )

# Transform into Hugging Face Dataset
dataset = Dataset.from_pandas(df)

# 3. Whisper 모델 불러오기
model_name = "openai/whisper-small"
processor = WhisperProcessor.from_pretrained(model_name)
model = WhisperForConditionalGeneration.from_pretrained(model_name)

# 4. 토큰화 적용
dataset = dataset.map(preprocess_function, batched=True)

def compute_metrics(pred):
    pred_ids = pred.predictions.argmax(-1)
    label_ids = pred.label_ids
    pred_texts = processor.batch_decode(pred_ids, skip_special_tokens=True)
    label_texts = processor.batch_decode(label_ids, skip_special_tokens=True)
    accuracy = sum([1 if p == l else 0 for p, l in zip(pred_texts, label_texts)]) / len(label_texts)
    return {"accuracy": accuracy}

# 5. 학습 설정
training_args = TrainingArguments(
    output_dir="./whisper_model",
    evaluation_strategy="epoch",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    save_strategy="steps",
    save_steps=5000,
    logging_dir="./logs",
    learning_rate=1e-5,
    weight_decay=0.01,
    num_train_epochs=3,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    eval_dataset=dataset,
    tokenizer=processor,
    compute_metrics=compute_metrics
)

# 6. 모델 학습
trainer.train()

# 7. 모델 저장
model.save_pretrained("./whisper_finetuned")
processor.save_pretrained("./whisper_finetuned")